# Notebook 06 — Pre-trained Embeddings (Section 5)

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings, pickle
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score, f1_score

import nltk
nltk.download('punkt', quiet=True)
from nltk.tokenize import word_tokenize
from collections import Counter
from tqdm import tqdm

OUTPUTS_DIR = Path("outputs")
MODELS_DIR = Path("models")
LOGS_DIR = Path("logs")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Device: cpu


## 1. Load Data & Vocabulary (same as Notebook 05)

In [2]:
train_df = pd.read_csv(OUTPUTS_DIR / 'train.csv').dropna(subset=['text', 'sentiment'])
test_df = pd.read_csv(OUTPUTS_DIR / 'test.csv').dropna(subset=['text', 'sentiment'])

MAX_VOCAB = 20000
MAX_SEQ_LEN = 100
PAD_TOKEN = '<PAD>'
UNK_TOKEN = '<UNK>'
EMBED_DIM = 100

def tokenize(text):
    return word_tokenize(str(text).lower())

all_tokens = []
for text in tqdm(train_df['text'], desc="Tokenizing"):
    all_tokens.extend(tokenize(text))

counter = Counter(all_tokens)
vocab_words = [PAD_TOKEN, UNK_TOKEN] + [w for w, _ in counter.most_common(MAX_VOCAB - 2)]
word2idx = {w: i for i, w in enumerate(vocab_words)}

def encode(text, max_len=MAX_SEQ_LEN):
    tokens = tokenize(text)[:max_len]
    ids = [word2idx.get(t, word2idx[UNK_TOKEN]) for t in tokens]
    ids += [word2idx[PAD_TOKEN]] * (max_len - len(ids))
    return ids

print(f"Vocabulary size: {len(word2idx)}")

Tokenizing: 100%|██████████| 19282/19282 [00:02<00:00, 7869.47it/s]

Vocabulary size: 20000


## 2. Load GloVe Embeddings

In [3]:
GLOVE_PATH = OUTPUTS_DIR / 'glove.6B.100d.txt'

glove_vectors = {}
if GLOVE_PATH.exists():
    with open(GLOVE_PATH, 'r', encoding='utf-8') as f:
        for line in tqdm(f, desc="Loading GloVe"):
            parts = line.split()
            glove_vectors[parts[0]] = np.array(parts[1:], dtype=np.float32)
    print(f"Loaded {len(glove_vectors)} GloVe vectors")
else:
    print("GloVe file not found. Run Notebook 03 first to download it.")

# Build embedding matrix
embed_matrix = np.zeros((len(word2idx), EMBED_DIM), dtype=np.float32)
found = 0
for word, idx in word2idx.items():
    if word in glove_vectors:
        embed_matrix[idx] = glove_vectors[word]
        found += 1
    else:
        embed_matrix[idx] = np.random.normal(0, 0.1, EMBED_DIM)

print(f"GloVe coverage: {found}/{len(word2idx)} words ({100*found/len(word2idx):.1f}%)")

Loading GloVe: 400000it [00:03, 101449.85it/s]


Loaded 400000 GloVe vectors
GloVe coverage: 14467/20000 words (72.3%)


## 3. Dataset & Model

In [4]:
class ReviewDataset(Dataset):
    def __init__(self, df, label_col, le=None):
        self.texts = [encode(t) for t in tqdm(df['text'], desc="Encoding")]
        if le is None:
            self.le = LabelEncoder()
            self.labels = self.le.fit_transform(df[label_col].astype(str))
        else:
            self.le = le
            self.labels = le.transform(df[label_col].astype(str))
    
    def __len__(self): return len(self.texts)
    
    def __getitem__(self, idx):
        return torch.tensor(self.texts[idx], dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.long)
    
    def num_classes(self): return len(self.le.classes_)


class PretrainedEmbedClassifier(nn.Module):
    def __init__(self, embed_matrix, num_classes, freeze=True):
        super().__init__()
        vocab_size, embed_dim = embed_matrix.shape
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.embedding.weight = nn.Parameter(torch.tensor(embed_matrix), requires_grad=not freeze)
        self.dropout = nn.Dropout(0.3)
        self.fc1 = nn.Linear(embed_dim, 128)
        self.fc2 = nn.Linear(128, num_classes)
        self.relu = nn.ReLU()
    
    def forward(self, x):
        emb = self.embedding(x).mean(dim=1)
        out = self.relu(self.fc1(self.dropout(emb)))
        return self.fc2(self.dropout(out))

## 4. Train — Frozen vs Fine-tuned GloVe

In [5]:
def train_model(task_name, label_col, freeze, num_epochs=10, batch_size=64, lr=1e-3):
    mode = 'frozen' if freeze else 'finetuned'
    print(f"\n{'='*60}\nTask: {task_name} | GloVe: {mode}\n{'='*60}")
    
    train_ds = ReviewDataset(train_df, label_col)
    test_ds = ReviewDataset(test_df, label_col, le=train_ds.le)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size)
    
    model = PretrainedEmbedClassifier(embed_matrix, train_ds.num_classes(), freeze=freeze).to(DEVICE)
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    criterion = nn.CrossEntropyLoss()
    writer = SummaryWriter(log_dir=str(LOGS_DIR / f'pretrained_{task_name}_{mode}'))
    
    best_f1 = 0
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        for texts, labels in train_loader:
            texts, labels = texts.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(texts), labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for texts, labels in test_loader:
                all_preds.extend(model(texts.to(DEVICE)).argmax(1).cpu().numpy())
                all_labels.extend(labels.numpy())
        
        acc = accuracy_score(all_labels, all_preds)
        f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
        avg_loss = total_loss / len(train_loader)
        writer.add_scalar('Loss/train', avg_loss, epoch)
        writer.add_scalar('Accuracy/test', acc, epoch)
        writer.add_scalar('F1_macro/test', f1, epoch)
        print(f"Epoch {epoch+1}/{num_epochs} | Loss: {avg_loss:.4f} | Acc: {acc:.4f} | F1: {f1:.4f}")
        
        if f1 > best_f1:
            best_f1 = f1
            torch.save(model.state_dict(), MODELS_DIR / f'pretrained_{task_name}_{mode}.pt')
    
    # Export embeddings
    embed_weights = model.embedding.weight.data.cpu()[:500]
    labels_tb = [vocab_words[i] for i in range(500)]
    writer.add_embedding(embed_weights, metadata=labels_tb, tag=f'glove_{mode}_{task_name}')
    writer.close()
    
    print(f"Best F1: {best_f1:.4f}")
    return {'task': task_name, 'mode': mode, 'accuracy': acc, 'f1_macro': best_f1}

## 5. Run Experiments

In [6]:
all_results = []
all_results.append(train_model('sentiment', 'sentiment', freeze=True))
all_results.append(train_model('sentiment', 'sentiment', freeze=False))

results_df = pd.DataFrame(all_results)
print(results_df)
results_df.to_csv(OUTPUTS_DIR / 'results_pretrained.csv', index=False)
print("\nTensorBoard: tensorboard --logdir=logs/")


Task: sentiment | GloVe: frozen


Encoding: 100%|██████████| 4821/4821 [00:00<00:00, 7634.87it/s]


Epoch 1/10 | Loss: 0.8267 | Acc: 0.7173 | F1: 0.5148
Epoch 2/10 | Loss: 0.7347 | Acc: 0.7521 | F1: 0.5383
Epoch 3/10 | Loss: 0.7259 | Acc: 0.7536 | F1: 0.5400
Epoch 4/10 | Loss: 0.7148 | Acc: 0.7488 | F1: 0.5368
Epoch 5/10 | Loss: 0.7109 | Acc: 0.7517 | F1: 0.5387
Epoch 6/10 | Loss: 0.7092 | Acc: 0.7559 | F1: 0.5417
Epoch 7/10 | Loss: 0.7049 | Acc: 0.7625 | F1: 0.5461
Epoch 8/10 | Loss: 0.7033 | Acc: 0.7600 | F1: 0.5445
Epoch 9/10 | Loss: 0.7004 | Acc: 0.7606 | F1: 0.5447
Epoch 10/10 | Loss: 0.6982 | Acc: 0.7557 | F1: 0.5416
Best F1: 0.5461

Task: sentiment | GloVe: finetuned


Encoding: 100%|██████████| 4821/4821 [00:00<00:00, 7741.85it/s]


Epoch 1/10 | Loss: 0.7583 | Acc: 0.7812 | F1: 0.5594
Epoch 2/10 | Loss: 0.5979 | Acc: 0.7953 | F1: 0.5696
Epoch 3/10 | Loss: 0.5618 | Acc: 0.8009 | F1: 0.5736
Epoch 4/10 | Loss: 0.5416 | Acc: 0.7982 | F1: 0.5720
Epoch 5/10 | Loss: 0.5240 | Acc: 0.8048 | F1: 0.5783
Epoch 6/10 | Loss: 0.5083 | Acc: 0.8052 | F1: 0.5921
Epoch 7/10 | Loss: 0.5004 | Acc: 0.8079 | F1: 0.5902
Epoch 8/10 | Loss: 0.4870 | Acc: 0.8058 | F1: 0.5992
Epoch 9/10 | Loss: 0.4725 | Acc: 0.8073 | F1: 0.5947
Epoch 10/10 | Loss: 0.4646 | Acc: 0.8046 | F1: 0.5919
Best F1: 0.5992
        task       mode  accuracy  f1_macro
0  sentiment     frozen  0.755652  0.546100
1  sentiment  finetuned  0.804605  0.599231

TensorBoard: tensorboard --logdir=logs/


## Summary
- Initialized embeddings from GloVe 100d
- Compared frozen vs fine-tuned embeddings on sentiment task
- TensorBoard embedding visualization exported